## EOS vs MAT, MAP thresholds


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

GIMMS_PHENOLOGY_DIR = "/Users/xingyihuang/Jupyter Code/Phenology/Appeal/data/satellite_data/images/PKU-GIMMS/phenology"
GIMMS_PHENOLOGY_PATTERN = "GIMMS_Phenology_SnowFilter_Forest1114_{year}.tif"

def load_gimms_snowfilter_sos_eos(df, years, phenology_dir=GIMMS_PHENOLOGY_DIR):
        coords = list(zip(df["longitude"].values, df["latitude"].values))
    n = len(coords)
    sample_year = next(
        y for y in years
        if os.path.exists(os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=y)))
    )
    with rasterio.open(os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=sample_year))) as src:
        transform = src.transform
        height, width = src.height, src.width

    rows = np.empty(n, dtype=np.int32)
    cols = np.empty(n, dtype=np.int32)
    for i, (lon, lat) in enumerate(coords):
        r, c = rasterio.transform.rowcol(transform, lon, lat)
        rows[i], cols[i] = r, c

    valid_rc = (rows >= 0) & (cols >= 0) & (rows < height) & (cols < width)

    for year in years:
        fp = os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=year))
        if not os.path.exists(fp):
            print(f"  missing phenology file: {fp}", flush=True)
            df[f"sos_{year}"] = np.nan
            df[f"eos_{year}"] = np.nan
            continue
        with rasterio.open(fp) as src:
            sos_band = src.read(1)
            eos_band = src.read(2)
        sos = np.full(n, np.nan, dtype=np.float32)
        eos = np.full(n, np.nan, dtype=np.float32)
        sos[valid_rc] = sos_band[rows[valid_rc], cols[valid_rc]]
        eos[valid_rc] = eos_band[rows[valid_rc], cols[valid_rc]]
        sos[~np.isfinite(sos)] = np.nan
        eos[~np.isfinite(eos)] = np.nan
        df[f"sos_{year}"] = sos
        df[f"eos_{year}"] = eos
    return df

def _load_annual_climate_for_coords(lons, lats, years):
        key = set(zip(np.round(lons, 5), np.round(lats, 5)))
    years = set(int(y) for y in years)
    coords = pd.DataFrame({"longitude": lons, "latitude": lats})
    coords["_k"] = list(zip(np.round(coords["longitude"], 5), np.round(coords["latitude"], 5)))

    def load_chunked(paths, prefix):
        hits = []
        for fp in paths:
            cols = pd.read_csv(fp, nrows=0).columns
            ycols = [c for c in cols if c.startswith(prefix) and int(c.split("_")[-1]) in years]
            if not ycols:
                continue
            usecols = ["longitude", "latitude"] + ycols
            for chunk in pd.read_csv(fp, usecols=usecols, chunksize=200_000):
                mask = [
                    (round(lo, 5), round(la, 5)) in key
                    for lo, la in zip(chunk["longitude"], chunk["latitude"])
                ]
                sub = chunk.loc[mask]
                if len(sub):
                    hits.append(sub)
        if not hits:
            return coords[["longitude", "latitude"]].copy()
        df = pd.concat(hits, ignore_index=True)
        df["_k"] = list(zip(np.round(df["longitude"], 5), np.round(df["latitude"], 5)))
        ycols = [c for c in df.columns if c.startswith(prefix)]
        df = df.groupby("_k", as_index=False)[ycols].first()
        return coords[["_k"]].merge(df, on="_k", how="left")

    t_paths = [
        "../../data/climate_data/tables/climate_data/temp/temp-1982-1999.csv",
        "../../data/climate_data/tables/climate_data/temp/temp-2000-2024.csv",
    ]
    p_paths = [
        "../../data/climate_data/tables/climate_data/prcp/prcp-1982-1999.csv",
        "../../data/climate_data/tables/climate_data/prcp/prcp-2000-2024.csv",
    ]
    print("  loading annual T from climate tables...", flush=True)
    tdf = load_chunked(t_paths, "annual_t_")
    print("  loading annual P from climate tables...", flush=True)
    pdf = load_chunked(p_paths, "annual_p_")
    out = coords[["longitude", "latitude"]].copy()
    for c in tdf.columns:
        if c.startswith("annual_t_"):
            out[c] = tdf[c].values
    for c in pdf.columns:
        if c.startswith("annual_p_"):
            out[c] = pdf[c].values
    return out

def read_satellite_data(veg_type, satellite):
    veg_class = pd.read_csv("../../data/veg_class_data/tables/veg_class.csv")
    clim_fp = f"../../data/satellite_data/tables/phenology_climate/{satellite}.csv"
    if satellite == "gimms" and not os.path.exists(clim_fp):
        print(f"  {clim_fp} missing — rebuilding annual T/P from climate tables", flush=True)
        forest = veg_class[veg_class["veg_class"].isin([12, 13, 14])].copy()
        if veg_type in (12, 13, 14):
            forest = forest[forest["veg_class"] == veg_type].copy()
        years = list(range(1982, 2023))
        df_satellite = _load_annual_climate_for_coords(
            forest["longitude"].values, forest["latitude"].values, years
        )
        df = forest.merge(df_satellite, on=["longitude", "latitude"], how="inner")
    else:
        df_satellite = pd.read_csv(clim_fp)
        df = pd.merge(df_satellite, veg_class, on=["latitude", "longitude"], how="inner")

    if veg_type in (12, 13, 14):
        df = df[df["veg_class"].isin([veg_type])]
    else:
        df = df[df["veg_class"].isin([12, 13, 14])]

    eos_cols = [col for col in df.columns if "eos" in col]
    t_cols = [col for col in df.columns if "annual_t" in col]
    p_cols = [col for col in df.columns if "annual_p" in col]
    sos_cols = [col for col in df.columns if "sos" in col]

    if satellite == "gimms":
        years = [str(y) for y in range(1982, 2023)]
        df = df.drop(columns=[c for c in eos_cols + sos_cols], errors="ignore")
        df = load_gimms_snowfilter_sos_eos(df, [int(y) for y in years])
        eos_cols = [col for col in df.columns if col.startswith("eos_")]
        sos_cols = [col for col in df.columns if col.startswith("sos_")]
    elif satellite == "avhrr":
        years = [str(y) for y in range(1982, 2017)]
    elif satellite == "modis":
        years = [str(y) for y in range(2001, 2024)]
        mask_sos = (df[sos_cols] < 0).any(axis=1)
        mask_eos = (df[eos_cols] > 365).any(axis=1)
        df = df[~(mask_sos | mask_eos)].copy()
    else:
        years = [str(y) for y in range(2013, 2023)]
        mask_sos = (df[sos_cols] < 0).any(axis=1)
        mask_eos = (df[eos_cols] > 365).any(axis=1)
        df = df[~(mask_sos | mask_eos)].copy()

    cols = years
    df = df[[col for col in eos_cols + t_cols + p_cols + sos_cols if any(y in col for y in cols)] + ["latitude", "longitude", "veg_class"]].copy()
    t_cols_df = [col for col in df.columns if "annual_t" in col]
    df[t_cols_df] = df[t_cols_df] - 273.5  # Convert temperature
    df.columns = df.columns.str.replace(r"\D*(\d{4})$", lambda m: f"{m.group(0)[0:-4]}{m.group(1)}", regex=True)
    df["annual_t"] = df[[col for col in df.columns if "annual_t" in col]].mean(axis=1)
    df["annual_p"] = df[[col for col in df.columns if "annual_p" in col]].mean(axis=1)
    eos_year_cols = [col for col in df.columns if col.startswith("eos_")]
    sos_year_cols = [col for col in df.columns if col.startswith("sos_")]
    df["eos"] = df[eos_year_cols].mean(axis=1)
    df["sos"] = df[sos_year_cols].mean(axis=1)
    if satellite == "gimms":
        df = df[df["eos"].notna()].copy()
    return df


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress, f

satellite = "gimms"
veg_type = 0
df = read_satellite_data(veg_type, satellite)
df = df[
    (df["annual_t"] >= -20) & (df["annual_t"] <= 20) &
    (df["annual_p"] >= 0) & (df["annual_p"] <= 4)
]
min_bin_count = 20

def compute_temp_precip_grid(x_mat, y_map_mm, z_eos, min_z_count,
                             x_edges=None, y_edges=None):
    if x_edges is None:
        x_edges = np.arange(-7, 20.25, 0.25)        # 0.25°C bins; last is right bound
    if y_edges is None:
        y_edges = np.arange(300, 1850, 100)

    n_x = len(x_edges) - 1
    n_y = len(y_edges) - 1
    z_mean = np.full((n_x, n_y), np.nan)
    z_std = np.full((n_x, n_y), np.nan)
    z_count = np.zeros((n_x, n_y), dtype=int)

    for i in range(n_x):
        for j in range(n_y):
            mask = (
                (x_mat >= x_edges[i]) & (x_mat < x_edges[i + 1]) &
                (y_map_mm >= y_edges[j]) & (y_map_mm < y_edges[j + 1])
            )
            n = int(np.sum(mask))
            if n >= min_z_count:
                z_mean[i, j] = np.nanmean(z_eos[mask])
                z_std[i, j] = np.nanstd(z_eos[mask])
                z_count[i, j] = n

    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])
    return z_mean, z_std, z_count, x_centers, y_centers, x_edges, y_edges

def find_breakpoint_on_means(mids, means, min_seg=10, sse_improve_min=0.015, alpha=0.05):
    mids = np.asarray(mids, dtype=float)
    means = np.asarray(means, dtype=float)
    ok = np.isfinite(mids) & np.isfinite(means)
    mids, means = mids[ok], means[ok]
    if len(mids) < 2 * min_seg:
        return np.nan, False

    peak = float(mids[int(np.nanargmax(means))])
    candidates = mids[mids >= peak]
    best_sse, best_bp = np.inf, np.nan
    for bp in candidates:
        left = mids <= bp
        right = mids >= bp
        if left.sum() < min_seg or right.sum() < min_seg:
            continue
        s1, i1, *_ = linregress(mids[left], means[left])
        s2, i2, *_ = linregress(mids[right], means[right])
        pred = np.empty_like(means)
        pred[left] = s1 * mids[left] + i1
        pred[right] = s2 * mids[right] + i2
        both = left & right
        if both.any():
            pred[both] = 0.5 * ((s1 * mids[both] + i1) + (s2 * mids[both] + i2))
        sse = np.sum((means - pred) ** 2)
        if sse < best_sse:
            best_sse, best_bp = sse, float(bp)

    if not np.isfinite(best_bp):
        return np.nan, False

    left = mids <= best_bp
    right = mids >= best_bp
    s1, i1, *_ = linregress(mids[left], means[left])
    s2, i2, *_ = linregress(mids[right], means[right])
    pred_p = np.empty_like(means)
    pred_p[left] = s1 * mids[left] + i1
    pred_p[right] = s2 * mids[right] + i2
    both = left & right
    if both.any():
        pred_p[both] = 0.5 * ((s1 * mids[both] + i1) + (s2 * mids[both] + i2))
    sse_p = float(np.sum((means - pred_p) ** 2))

    s_s, i_s, *_ = linregress(mids, means)
    sse_s = float(np.sum((means - (s_s * mids + i_s)) ** 2))
    N = len(mids)
    df2 = N - 4
    if df2 <= 0 or sse_p <= 0 or sse_s <= 0:
        return best_bp, False
    Fstat = ((sse_s - sse_p) / 2) / (sse_p / df2)
    p_val = 1 - f.cdf(Fstat, 2, df2)
    sse_improvement = (sse_s - sse_p) / sse_s
    return best_bp, bool(p_val < alpha and sse_improvement >= sse_improve_min)

def format_p(p):
    if not np.isfinite(p):
        return "p=NA"
    if p < 0.01:
        return "p<0.01"
    if p < 0.05:
        return "p<0.05"
    return f"p={p:.2f}"

def plot_eos_mat_from_grid(
    df,
    temp_col="annual_t",
    precip_col="annual_p",
    eos_col="eos",
    min_z_count=20,
    thresholds_mm=None,
):
    if thresholds_mm is None:
        thresholds_mm = list(range(700, 1401, 100))

    x = df[temp_col].to_numpy(dtype=float)
    y_mm = df[precip_col].to_numpy(dtype=float) * 1000.0
    z = df[eos_col].to_numpy(dtype=float)

    z_mean, z_std, z_count, x_c, y_c, x_edges, y_edges = compute_temp_precip_grid(
        x, y_mm, z, min_z_count=min_z_count
    )

    n_cols = 4
    n_rows = int(np.ceil(len(thresholds_mm) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.6 * n_rows), sharey=True)
    axes = np.atleast_1d(axes).ravel()

    rows = []
    for ax, T in zip(axes, thresholds_mm):
        js = [j for j in range(len(y_edges) - 1) if y_edges[j + 1] <= T]
        label = f"MAP < {T:.0f} mm"
        ax.set_ylim(240, 320)
        ax.set_yticks(np.arange(240, 321, 20))
        ax.tick_params(labelsize=12)
        ax.text(0.02, 0.98, label, transform=ax.transAxes, ha="left", va="top", fontsize=14)

        if not js:
            ax.text(0.5, 0.5, "no MAP columns", transform=ax.transAxes,
                    ha="center", va="center", color="gray")
            continue

        mids, yline, yerr, n_pix_list = [], [], [], []
        for i, xc in enumerate(x_c):
            m = z_mean[i, js]
            s = z_std[i, js]
            w = z_count[i, js].astype(float)
            ok = np.isfinite(m) & (w > 0)
            if ok.sum() == 0:
                continue
                        mu = np.average(m[ok], weights=w[ok])
            if ok.sum() == 1:
                sd = float(s[ok][0]) if np.isfinite(s[ok][0]) else np.nan
            else:
                var = np.nansum(w[ok] * (np.nan_to_num(s[ok], nan=0.0) ** 2 + (m[ok] - mu) ** 2)) / np.sum(w[ok])
                sd = float(np.sqrt(max(var, 0.0)))
            mids.append(xc)
            yline.append(mu)
            yerr.append(sd)
            n_pix_list.append(int(np.sum(w[ok])))

        mids = np.asarray(mids, dtype=float)
        yline = np.asarray(yline, dtype=float)
        yerr = np.asarray(yerr, dtype=float)

        if len(mids) < 2:
            ax.text(0.5, 0.5, "insufficient grid cells", transform=ax.transAxes,
                    ha="center", va="center", color="gray")
            continue

        ax.fill_between(mids, yline - yerr, yline + yerr, color="#e03c31", alpha=0.15)
        ax.plot(mids, yline, color="#e03c31", lw=2, alpha=0.9)

        bp, use_bp = find_breakpoint_on_means(mids, yline, min_seg=10)

        if use_bp and np.isfinite(bp):
            ax.axvline(bp, color="#e4542a", ls="-", lw=1.5, alpha=0.5)
            ax.text(bp + 0.8, 312, f"{bp:.2f}°C", color="#e4542a", fontsize=12,
                    ha="left", va="top", weight="bold")
            below = mids <= bp
            above = mids >= bp
            slope1 = slope2 = p1 = p2 = np.nan
            if below.sum() >= 2:
                slope1, i1, r1, p1, _ = linregress(mids[below], yline[below])
                xf = np.linspace(mids[below].min(), mids[below].max(), 50)
                ax.plot(xf, slope1 * xf + i1, ls="--", color="#e4542a", lw=2)
            if above.sum() >= 2:
                slope2, i2, r2, p2, _ = linregress(mids[above], yline[above])
                xf = np.linspace(mids[above].min(), mids[above].max(), 50)
                ax.plot(xf, slope2 * xf + i2, ls="--", color="#e4542a", lw=2)
            ax.text(0.02, 0.02, f"L: {slope1:.2f}\n{format_p(p1)}",
                    transform=ax.transAxes, fontsize=12, ha="left", va="bottom")
            ax.text(0.98, 0.02, f"R: {slope2:.2f}\n{format_p(p2)}",
                    transform=ax.transAxes, fontsize=12, ha="right", va="bottom")
            fit = "breakpoint"
            lin_slope = np.nan
        else:
            slope, intercept, r_val, p_val, _ = linregress(mids, yline)
            xf = np.linspace(mids.min(), mids.max(), 50)
            ax.plot(xf, slope * xf + intercept, ls="--", color="#e4542a", lw=2)
            ax.text(0.02, 0.02, f"Linear\nSlope={slope:.2f}\n{format_p(p_val)}",
                    transform=ax.transAxes, fontsize=12, ha="left", va="bottom")
            fit = "linear"
            bp = np.nan
            slope1 = slope2 = np.nan
            lin_slope = slope

        rows.append({
            "threshold": label,
            "n_map_cols": len(js),
            "n_mat_bins": len(mids),
            "n_pixels": int(np.sum(n_pix_list)),
            "fit": fit,
            "bp_C": bp,
            "left_slope": slope1,
            "right_slope": slope2,
            "linear_slope": lin_slope,
        })
        ax.set_xlabel("MAT (°C)", fontsize=14)

    for j, ax in enumerate(axes):
        if j >= len(thresholds_mm):
            fig.delaxes(ax)
        elif j % n_cols == 0:
            ax.set_ylabel("EOS (DOY)", fontsize=14)

    plt.tight_layout()
    return fig, pd.DataFrame(rows)

thresholds_mm = list(range(700, 1401, 100))  # <700, <800, ... <1400

fig_grid, grid_summary = plot_eos_mat_from_grid(
    df,
    min_z_count=min_bin_count,
    thresholds_mm=thresholds_mm,
)
print(grid_summary.to_string(index=False))
out_fp = f"../../results/si_figures/si_fig1/{satellite}_eos_mat_map_lt_thresholds.png"
fig_grid.savefig(out_fp, dpi=500, bbox_inches="tight")
print(f"Saved {out_fp}")
